# Estimacion de velocidad de flujo libre con jams

Este cuaderno calcula estimaciones de velocidad de flujo libre por tramo usando los eventos de jams procesados.

- lee los archivos parquet de jams en lotes para evitar cargar todo en memoria;
- normaliza columnas claves (velocidad promedio, delay, conteo de jams);
- ajusta el tiempo de recorrido removiendo el delay medio y calcula la velocidad de flujo libre por evento;
- agrega estadisticos por tramo (mediana, p85, media) y exporta los resultados.

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
from typing import Iterable, Sequence

import numpy as np
import pandas as pd
import geopandas as gpd

try:
    import pyarrow.parquet as pq
except ImportError:  # pyarrow habilita la lectura por lotes, pero no es obligatorio
    pq = None

try:
    from tqdm.auto import tqdm
except ImportError:  # tqdm solo se usa para la barra de progreso
    def tqdm(iterable, **kwargs):
        return iterable

pd.options.display.max_columns = 60
pd.options.display.float_format = "{:.3f}".format

In [ ]:
PROJECT_ROOT = Path("..").resolve()
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "datos_procesados"
JAMS_DIR = PROCESSED_DATA_DIR / "jams_procesados"
ROADS_FILE = PROJECT_ROOT / "data" / "2_Estructura_via" / "estructura_via_completa" / "MVI.geojson"
OUTPUT_DIR = PROCESSED_DATA_DIR / "features"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FREE_FLOW_PARQUET = OUTPUT_DIR / "free_flow_speed_estimates.parquet"
FREE_FLOW_PICKLE = OUTPUT_DIR / "free_flow_speed_estimates.pkl"

PARQUET_BATCH_SIZE = 200_000  # filas por lote (None usa el size del row group)

print(f"PROCESSED_DATA_DIR: {PROCESSED_DATA_DIR}")
print(f"JAMS_DIR: {JAMS_DIR}")
print(f"ROADS_FILE: {ROADS_FILE}")
print(f"Batch size (rows): {PARQUET_BATCH_SIZE}")

In [ ]:
def discover_schema_columns(parquet_path: Path) -> Sequence[str]:
    if pq is not None:
        return pq.read_schema(parquet_path).names
    temp_df = pd.read_parquet(parquet_path)
    return temp_df.columns.tolist()


def select_column(columns: Sequence[str], candidates: Sequence[str], label: str) -> str:
    for candidate in candidates:
        if candidate in columns:
            return candidate
    raise KeyError(f"No se encontro columna para {label}. Revisar schema: {columns}")


def build_mvi_identifier(df: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    df = df.copy()
    df["road_id"] = (
        df["MVICIV"].astype(str).str.strip()
        + "_"
        + df["MVISVIA"].astype(str).str.replace(" ", "", regex=False)
    )
    df["segment_idx"] = df.groupby("road_id").cumcount() + 1
    df["road_id"] = df["road_id"] + "." + df["segment_idx"].astype(str)
    return df


def iterate_parquet_batches(
    path: Path,
    columns: Sequence[str],
    batch_size: int | None = None,
) -> Iterable[pd.DataFrame]:
    if pq is None:
        yield pd.read_parquet(path, columns=columns)
        return
    parquet_file = pq.ParquetFile(path)
    for batch in parquet_file.iter_batches(columns=columns, batch_size=batch_size):
        yield batch.to_pandas()


def build_jams_column_mapping(schema_cols: Sequence[str]) -> tuple[dict[str, str], bool]:
    road_col = select_column(schema_cols, ["road_id", "id_mvi"], "road_id")
    time_col = select_column(schema_cols, ["timeframe", "timestamp"], "timeframe")
    speed_col = select_column(
        schema_cols,
        ["vel_mean", "speed_mean", "speedKMH", "speedKMH_mean", "speed_kmh_mean"],
        "velocidad media",
    )
    delay_col = select_column(
        schema_cols,
        ["delay_sum", "delay_total", "delay", "delay_mean"],
        "delay agregado",
    )
    jam_candidates = ["count_jams", "jam_id_counts", "jam_count", "jam_events"]
    jam_col = next((c for c in jam_candidates if c in schema_cols), None)

    rename_map = {
        road_col: "road_id",
        time_col: "timeframe",
        speed_col: "vel_mean",
        delay_col: "delay_total",
    }
    has_jam_events = jam_col is not None
    if has_jam_events:
        rename_map[jam_col] = "jam_events"
    return rename_map, has_jam_events


def iter_normalized_chunks(
    parquet_path: Path,
    rename_map: dict[str, str],
    has_jam_events: bool,
    batch_size: int | None,
) -> Iterable[pd.DataFrame]:
    columns = list(rename_map.keys())
    for chunk in iterate_parquet_batches(parquet_path, columns=columns, batch_size=batch_size):
        chunk = chunk.rename(columns=rename_map)
        if not has_jam_events:
            chunk["jam_events"] = 1.0
        yield chunk

In [ ]:
roads_gdf = gpd.read_file(ROADS_FILE)[["MVICIV", "MVISVIA", "geometry"]]
roads_gdf = build_mvi_identifier(roads_gdf)
roads_gdf = roads_gdf.explode(ignore_index=True).dissolve(by="road_id").reset_index()
roads_gdf = roads_gdf.to_crs(3116)
roads_gdf["segment_length_m"] = roads_gdf.geometry.length

roads_lengths = roads_gdf[["road_id", "segment_length_m"]].copy()
roads_lengths["segment_length_km"] = roads_lengths["segment_length_m"] / 1000.0
length_map = roads_lengths.set_index("road_id")["segment_length_m"].to_dict()

del roads_gdf

print(f"Tramos en MVI: {len(roads_lengths):,}")
roads_lengths.head()

In [ ]:
jams_files = sorted(JAMS_DIR.glob("*.parquet"))
if not jams_files:
    raise FileNotFoundError(f"No se encontraron archivos parquet en {JAMS_DIR}")

print(f"Archivos de jams detectados: {len(jams_files)}")
for sample_path in jams_files[:3]:
    print(f" - {sample_path.name}")

schema_cols = discover_schema_columns(jams_files[0])
rename_map, has_jam_events = build_jams_column_mapping(schema_cols)

print("Columnas mapeadas:")
for raw_col, std_col in rename_map.items():
    print(f"  {raw_col} -> {std_col}")
print(f"Columna de conteo de jams disponible: {has_jam_events}")

In [ ]:
aggregated_data: dict[str, dict[str, object]] = {}
total_rows = 0
total_chunks = 0

total_files = len(jams_files)
progress_bar = tqdm(
    enumerate(jams_files, start=1),
    total=total_files,
    desc="Archivos de jams",
    mininterval=5.0,
)

for file_idx, parquet_path in progress_bar:
    file_chunks = 0
    file_rows = 0
    for chunk in iter_normalized_chunks(
        parquet_path, rename_map, has_jam_events, PARQUET_BATCH_SIZE
    ):
        total_chunks += 1
        file_chunks += 1
        rows_in_chunk = len(chunk)
        total_rows += rows_in_chunk
        file_rows += rows_in_chunk
        if chunk.empty:
            continue

        chunk["road_id"] = chunk["road_id"].astype(str)
        chunk["timeframe"] = pd.to_datetime(
            chunk["timeframe"], utc=True, errors="coerce"
        )
        for col in ["vel_mean", "delay_total", "jam_events"]:
            chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

        chunk["segment_length_m"] = chunk["road_id"].map(length_map)
        chunk = chunk.dropna(
            subset=["segment_length_m", "vel_mean", "delay_total", "jam_events"]
        )
        chunk = chunk[
            (chunk["segment_length_m"] > 0)
            & (chunk["vel_mean"] > 0)
            & (chunk["jam_events"] > 0)
        ]
        if chunk.empty:
            continue

        chunk["delay_seconds_avg"] = chunk["delay_total"] / chunk["jam_events"]
        chunk.replace({np.inf: np.nan, -np.inf: np.nan}, inplace=True)
        chunk.dropna(subset=["delay_seconds_avg"], inplace=True)
        if chunk.empty:
            continue

        chunk["distance_km"] = chunk["segment_length_m"] / 1000.0
        chunk["travel_time_hours"] = chunk["distance_km"] / chunk["vel_mean"]
        chunk["delay_hours"] = chunk["delay_seconds_avg"] / 3600.0
        chunk["freeflow_time_hours"] = chunk["travel_time_hours"] - chunk["delay_hours"]
        chunk = chunk[chunk["freeflow_time_hours"] > 0]
        if chunk.empty:
            continue

        chunk["free_flow_speed_kmh"] = (
            chunk["distance_km"] / chunk["freeflow_time_hours"]
        )
        chunk.replace({np.inf: np.nan, -np.inf: np.nan}, inplace=True)
        chunk.dropna(subset=["free_flow_speed_kmh"], inplace=True)
        if chunk.empty:
            continue

        for road_id, group in chunk.groupby("road_id"):
            entry = aggregated_data.get(road_id)
            if entry is None:
                entry = {
                    "free_flow_values": [],
                    "jam_speed_sum": 0.0,
                    "jam_speed_count": 0,
                    "delay_sum": 0.0,
                    "delay_count": 0,
                    "timeframe_ids": set(),
                    "first_time": None,
                    "last_time": None,
                }
                aggregated_data[road_id] = entry

            entry["free_flow_values"].extend(group["free_flow_speed_kmh"].tolist())
            entry["jam_speed_sum"] += float(group["vel_mean"].sum())
            entry["jam_speed_count"] += int(len(group))
            entry["delay_sum"] += float(group["delay_seconds_avg"].sum())
            entry["delay_count"] += int(len(group))

            valid_timeframes = group["timeframe"].dropna()
            if not valid_timeframes.empty:
                timeframe_ints = valid_timeframes.astype("int64").to_numpy()
                entry["timeframe_ids"].update(timeframe_ints.tolist())
                first_val = timeframe_ints.min()
                last_val = timeframe_ints.max()
                if entry["first_time"] is None or first_val < entry["first_time"]:
                    entry["first_time"] = first_val
                if entry["last_time"] is None or last_val > entry["last_time"]:
                    entry["last_time"] = last_val

        if total_chunks % 25 == 0:
            progress_bar.set_postfix(
                {
                    "archivo": f"{file_idx}/{total_files}",
                    "chunks": total_chunks,
                    "filas": f"{total_rows:,}",
                },
                refresh=False,
            )

    if file_chunks == 0:
        tqdm.write(f"Archivo sin filas válidas: {parquet_path.name}")
    else:
        tqdm.write(
            f"Procesado {parquet_path.name}: {file_chunks} chunks, {file_rows:,} filas"
        )
    progress_bar.set_postfix(
        {
            "archivo": f"{file_idx}/{total_files}",
            "chunks": total_chunks,
            "filas": f"{total_rows:,}",
        },
        refresh=False,
    )

print(f"Total chunks procesados: {total_chunks}")
print(f"Total filas revisadas: {total_rows:,}")
print(f"Tramos con datos: {len(aggregated_data):,}")



In [ ]:
records: list[dict[str, object]] = []

for road_id, entry in aggregated_data.items():
    free_flow_values = np.asarray(entry["free_flow_values"], dtype=float)
    if free_flow_values.size == 0:
        continue

    record = {
        "road_id": road_id,
        "free_flow_speed_p50": float(np.median(free_flow_values)),
        "free_flow_speed_p85": float(np.quantile(free_flow_values, 0.85)),
        "free_flow_speed_mean": float(free_flow_values.mean()),
        "jam_speed_mean": (
            entry["jam_speed_sum"] / entry["jam_speed_count"]
            if entry["jam_speed_count"]
            else np.nan
        ),
        "delay_seconds_mean": (
            entry["delay_sum"] / entry["delay_count"]
            if entry["delay_count"]
            else np.nan
        ),
        "free_flow_samples": int(free_flow_values.size),
        "timeframes_with_jam": len(entry["timeframe_ids"]),
        "first_timeframe": (
            pd.to_datetime(entry["first_time"], utc=True)
            if entry["first_time"] is not None
            else pd.NaT
        ),
        "last_timeframe": (
            pd.to_datetime(entry["last_time"], utc=True)
            if entry["last_time"] is not None
            else pd.NaT
        ),
    }
    records.append(record)
    entry["free_flow_values"].clear()
    entry["timeframe_ids"].clear()

free_flow_per_road = pd.DataFrame.from_records(records)
print(f"Registros generados: {len(free_flow_per_road):,}")
free_flow_per_road.head()

In [ ]:
free_flow_summary = roads_lengths.merge(free_flow_per_road, on="road_id", how="left")
free_flow_summary["recommended_free_flow_kmh"] = free_flow_summary[
    "free_flow_speed_p85"
].fillna(free_flow_summary["free_flow_speed_p50"])

coverage_cols = [
    "road_id",
    "segment_length_km",
    "free_flow_speed_p50",
    "free_flow_speed_p85",
    "free_flow_speed_mean",
    "recommended_free_flow_kmh",
    "free_flow_samples",
    "timeframes_with_jam",
    "first_timeframe",
    "last_timeframe",
]
free_flow_summary[coverage_cols].head()

In [ ]:
free_flow_summary.to_parquet(FREE_FLOW_PARQUET, index=False)
free_flow_summary.to_pickle(FREE_FLOW_PICKLE)

print(f"Parquet exportado en: {FREE_FLOW_PARQUET}")
print(f"Pickle exportado en: {FREE_FLOW_PICKLE}")